![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 01 | Exploratory Data Analysis

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Understanding your data before you design anything. You cannot decide which tables you need until you know which columns repeat, which are broken, and which identify a row.

> **Done when:** you can name the 3+ tables you are going to build, you have written down at least 2 research questions, and you know every data-quality problem notebook 02 will have to fix.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [ ]:
%pip install openpyxl
%pip show openpyxl
%pip install python-dotenv

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

Run `python download_data.py <payments|airbnb|retail>` from the repo root first. Your files land in `data/raw/`, which is gitignored.

The data for this project came from https://cpds-data.org/data/

In [ ]:
# Puting all private info in extra file, so its not available for public and than retrieving it
from dotenv import load_dotenv
import os

load_dotenv()

data_path = os.getenv("DATA_PATH")

if data_path is None:
    raise ValueError("DATA_PATH is not set. Create a .env file first.")

In [ ]:
raw_data = data_path
print(data_path)

---
## 2. First look

Shape, column types, a handful of rows. What is one row of this data?

In [ ]:
df = pd.read_excel(raw_data)


In [ ]:
df.shape

In [ ]:
df.head()

---
## 3. Data quality

Missing values, duplicates, numbers stored as text, dates stored as strings. Write down what you find — this is the to-do list for notebook 02.

In [ ]:
#General info
df.info()

In [ ]:
#Missing values
missing_values = df.isna().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
#Duplicated values
df.duplicated().sum()

In [ ]:
#check if the most important columns for our research are unique
df.duplicated(subset=["country", "year"]).sum()

In [ ]:
#Data types
df.dtypes.value_counts()

In [ ]:
#Count number of distinct elements in country column
df["country"].nunique()

In [ ]:
#Check the counties spelling
df["country"].unique()

In [ ]:
#Check if the year range is what is announced in data description
df["year"].min(), df["year"].max()

In [ ]:
#Check if all countries have data for all years
df["year"].value_counts().sort_index()

In [ ]:
# Check if countries have country_id to use it as primary key for later. if not, then find the best candidate for this
import re

country_id_columns = [
    col for col in df.columns
    if re.search(r"country|id", str(col), re.IGNORECASE)
]

print(country_id_columns) #there is a countryn column wich we possibly could use as 
print(len(df["countryn"].unique()))#number of country looks like a good pretendent


Check what columns can be used as primary keys

In [ ]:
#check for primary key column for country and rename the column to country_id!!
df[["country","countryn"]]

---
## 4. Distributions

The numeric columns you care about, and the outliers you will have to decide about.

In [ ]:
numeric_columns = [
    "realgdpgr",
    "unemp",
    "inflation"
]

df[numeric_columns].describe()

In [ ]:
df["realgdpgr"].hist(bins=30)
#gdp data have a normal distribution slightly moved to the right

In [ ]:
df["unemp"].hist(bins=30)
#A histogram is right skewed 
#Right Skewed Distribution: Mode < Median < Mean

In [ ]:
df["inflation"].hist(bins=800)
plt.xlim(0, 40)

# histogram for inflation data shows a strongly right-skewed distribution
# most inflation observations appear concentrated roughly around 0–10%, while a small number of observations extend much farther to the right.

## GDP

GDP distribution looks normal, so no further investigation required at this stage

In [ ]:
df.boxplot(column="realgdpgr")

### Unemployment check in detail:

In [ ]:
df.boxplot(column="unemp")

In [ ]:
df["unemp"].describe()

In [ ]:
df.boxplot(column="unemp")

plt.ylabel("Unemployment (%)")
plt.title("Unemployment — Full Dataset")
plt.show()

In [ ]:
df.nlargest(10, "unemp")[
    ["country", "year", "unemp"]
]

In [ ]:
df.nsmallest(10, "unemp")[
    ["country", "year", "unemp"]
]

In [ ]:
# distribution/outlier analysis on the filtered dataset rather than the full historical dataset
df_period = df[df["year"].between(2000, 2019)].copy()

In [ ]:
df_period["unemp"].describe()

In [ ]:
df_period["unemp"].hist(bins=30)

plt.xlabel("Unemployment (%)")
plt.ylabel("Frequency")
plt.title("Distribution of Unemployment — 2000–2019")
plt.show()

In [ ]:
df_period.boxplot(column="unemp")

plt.ylabel("Unemployment (%)")
plt.title("Unemployment — 2000–2019")
plt.show()

In [ ]:
df_period.nlargest(10, "unemp")[
    ["country", "year", "unemp"]
]

### Inflation check in detail:

In [ ]:
df.boxplot(column="inflation")

In [ ]:
q1 = df["inflation"].quantile(0.25)
q3 = df["inflation"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [ ]:
inflation_outliers = df[
    (df["inflation"] < lower_bound) |
    (df["inflation"] > upper_bound)
]

inflation_outliers[["country", "year", "inflation"]]

In [ ]:
df.nlargest(10, "inflation")[
    ["country", "year", "inflation"]
]

In [ ]:
df_2000_2019 = df[
    df["year"].between(2000, 2019)
]

df_2000_2019.nlargest(10, "inflation")[
    ["country", "year", "inflation"]
]

In [ ]:
df_period["inflation"].hist(bins=50)
plt.xlabel("Inflation (%)")
plt.ylabel("Frequency")
plt.title("Inflation Distribution, 2000–2019")

In [ ]:
df_period.boxplot(column="inflation")
plt.title("Inflation, 2000–2019")

In [ ]:
df_period.nlargest(10, "inflation")[
    ["country", "year", "inflation"]
]

In [ ]:
df_period[
    (df_period["country"] == "Romania") &
    (df_period["year"].between(1999, 2002))
][["country", "year", "inflation"]]

The inflation variable is strongly right-skewed. Most observations between 2000 and 2019 are concentrated at relatively low inflation rates, while several country-year observations show substantially higher inflation. The boxplot identifies these observations as statistical outliers according to the IQR rule, but they are not necessarily data errors and should not be removed without further investigation.

--
After verifying the data with the world data bank, we can conclude that 45.8% inflation in romania is an accurate value(even though its unusual, so we can keep it), since previos peack of inflation in 1993 was even bigger
https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?locations=RO

---
## 5. Categorical columns

**This is the section that feeds tomorrow.** A column with few distinct values repeated over many rows is a lookup table waiting to happen. Find them.

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns

categorical_columns

In [ ]:
df[categorical_columns].nunique().sort_values()

In [ ]:
df["gov_party"].value_counts(dropna=False)

In [ ]:
df["country"].value_counts(dropna=False)

In [ ]:
df["gov_party"].nunique()

### Check up for extra columns:

Since the main focus of interest should be impact of country politics on the economy we are also going to check political data. Govermental parties are identified as numbers from 1 to 5
### Government Party Composition

### `gov_party`

Cabinet composition based on the **Schmidt Index**:

| Value | Description | Share of Left-Wing Parties |
|------:|-------------|----------------------------|
| 1 | Hegemony of right-wing and center parties | `gov_left1 = 0` |
| 2 | Dominance of right-wing and center parties | `0 < gov_left1 <= 33.33` |
| 3 | Balance of power between left and right | `33.33 < gov_left1 < 66.67` |
| 4 | Dominance of social-democratic and other left parties | `66.67 <= gov_left1 < 100` |
| 5 | Hegemony of social-democratic and other left parties | `gov_left = 100` |

### Coverage

- **Period covered:** 1960–2023

### Missing Values

Missing observations occur for:

- Bulgaria: 1993–1994
- Italy: 2012
- Romania: 2015–2016

These correspond to **full technocratic governments** or **partisan caretaker governments**.

### Source

Own calculations according to **Schmidt (1992)**.

In [ ]:
df["gov_party"].nunique()

In [ ]:
df["gov_party"].value_counts(dropna=False)

In [ ]:
df["gov_party"].unique()

In [ ]:
# for time period od interest
df_period["gov_party"].value_counts(dropna=False)


In [ ]:
df_period["gov_party"].value_counts(
    normalize=True,
    dropna=False
) * 100

Chcek related to goverment_party columns:

In [ ]:
df.columns[
    df.columns.str.contains(
        "gov.*left|gov.*cent|gov.*right|gov.*party",
        case=False,
        regex=True
    )
].tolist()

In [ ]:
df[
    ["country", "year", "gov_party",
     "gov_left1", "gov_cent1", "gov_right1"]
].head(20)

In [ ]:
df_period[
    ["country", "year", "gov_party",
     "gov_left1", "gov_cent1", "gov_right1"]
].head(20)

In [ ]:
df_period["gov_party"].value_counts().sort_index().plot(
    kind="bar"
)

plt.xlabel("Government Composition (Schmidt Index)")
plt.ylabel("Number of Country-Year Observations")
plt.title("Government Composition, 2000–2019")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df_period[df_period["gov_party"].isna()][
    ["country", "year", "gov_left1", "gov_cent1", "gov_right1"]
]

In [ ]:
political_columns = [
    "gov_left1",
    "gov_cent1",
    "gov_right1"
]

df[political_columns].describe()

In [ ]:
df[political_columns].isna().sum()


In [ ]:
df_period[political_columns].describe()

In [ ]:
#Check if all values are in percentual range:
for column in political_columns:
    invalid_values = df[
        (df[column] < 0) | (df[column] > 100)
    ]

    print(column, len(invalid_values))

In [ ]:
df[
    (df["gov_left1"] < 0) |
    (df["gov_left1"] > 100)
][
    ["country", "year", "gov_party",
     "gov_left1", "gov_cent1", "gov_right1"]
]

In [ ]:
#Is this going to impact our time period?
df_period[
    (df_period["gov_left1"] < 0) |
    (df_period["gov_left1"] > 100)
][
    ["country", "year", "gov_party",
     "gov_left1", "gov_cent1", "gov_right1"]
]

**Government composition range check**

The government composition variables were checked for values outside the expected
0–100% range. One out-of-range value was identified in `gov_left1` in the full
historical dataset. However, this observation falls outside the selected analysis
period (2000–2019) and therefore does not affect the final analytical dataset.

No out-of-range values were found for the selected period.

In [ ]:
government_total = (
    df_period["gov_left1"]
    + df_period["gov_cent1"]
    + df_period["gov_right1"]
)

government_total.describe()

In [ ]:
government_total.value_counts().sort_index().head(20)

In [ ]:
print("Below 99%:", (government_total < 99).sum())
print("99–101%:", government_total.between(99, 101).sum())
print("Above 101%:", (government_total > 101).sum())

In [ ]:
df_period.assign(
    government_total=government_total
).nsmallest(10, "government_total")[
    [
        "country",
        "year",
        "gov_party",
        "gov_left1",
        "gov_cent1",
        "gov_right1",
        "government_total"
    ]
]

---
## 6. Research questions

At least two, written here. Specific, answerable with this data, and with an answer you could be wrong about. For each one, say what you expect and why.

1. GDP growth and government composition
Research question:
How does average real GDP growth differ across government political compositions between 2000 and 2019?
Expectation:
I expect to observe some differences in average GDP growth between government composition categories, although I do not expect political composition alone to explain these differences because economic growth is influenced by many other factors.
2. Unemployment and government composition
Research question:
How does the unemployment rate differ across government political compositions between 2000 and 2019?
Expectation:
I expect unemployment rates to vary across government composition categories. However, I expect substantial variation within each category because unemployment is also influenced by country-specific conditions and broader economic cycles.
3. Inflation — optional but useful
Research question:
How does inflation differ across government political compositions between 2000 and 2019?
Expectation:
I expect inflation to show differences across government composition categories, but I also expect considerable variation between countries and years.
These work well because you could genuinely be wrong. For example, your SQL analysis might show virtually no meaningful difference in average GDP growth between the categories. That's still a valid result.

---
## 7. Into notebook 02

The tables you are going to build, and the problems you have to fix first.

Tables to build
1. countries — lookup table
- country_id — primary key
- country_name
- iso_code
2. government_types — lookup table
- government_type_id — primary key
- government_type
3. country_year — main analytical table
- observation_id — primary key
- country_id — foreign key → countries
- government_type_id — foreign key → government_types
- year
- left_percentage
- centre_percentage
- right_percentage
- gdp_growth
- unemployment
- inflation

Before loading the data into SQLite, I need to:
- select only the columns required for the analysis;
- filter the dataset to the selected period, 2000-2019;
- handle missing values in the selected political and economic variables;
- verify and correct data types where necessary;
- check duplicate country-year observations;
- check categorical values for inconsistent labels;
- investigate extreme values in GDP growth, unemployment and inflation before deciding whether they should be retained;
- transform repeated country and government-composition values into lookup tables;
- create primary and foreign keys for the relational database.

## Additional improvemets for data security
-added .env file where i store all the paths and keys so they are not available to public

## Info from the codebook for data:
https://cpds-data.org/wp-content/uploads/2025/09/codebook_cpds.pdf
## General Variables

| Variable | Description |
|----------|-------------|
| `year` | Year of observation |
| `country` | Country name |
| `countryn` | Country code number |

### Notes

- `year_1`, ..., `year_12` are auxiliary variables identical to `year`.
- `country_1`, ..., `country_12` are auxiliary variables identical to `country`.

### Country Codes

| Code | Country | Code | Country |
|------|---------|------|---------|
| 1 | Australia | 19 | Japan |
| 2 | Austria | 20 | Latvia |
| 3 | Belgium | 21 | Lithuania |
| 4 | Bulgaria | 22 | Luxembourg |
| 5 | Canada | 23 | Malta |
| 6 | Croatia | 24 | Netherlands |
| 7 | Cyprus (Greek part) | 25 | New Zealand |
| 8 | Czech Republic | 26 | Norway |
| 9 | Denmark | 27 | Poland |
| 10 | Estonia | 28 | Portugal |
| 11 | Finland | 29 | Romania |
| 12 | France | 30 | Slovakia |
| 13 | Germany | 31 | Slovenia |
| 14 | Greece | 32 | Spain |
| 15 | Hungary | 33 | Sweden |
| 16 | Iceland | 34 | Switzerland |
| 17 | Ireland | 35 | United Kingdom |
| 18 | Italy | 36 | USA |

### Selection of Government Composition Variables

The CPDS provides several measures of government composition and political
power. This analysis uses `gov_left1`, `gov_cent1`, and `gov_right1`, which
measure the percentage of cabinet posts held by left, centre, and right
parties, weighted by the number of days in office during each year.

These variables were selected because the main categorical variable,
`gov_party` (Schmidt Index), is based on cabinet composition and specifically
uses `gov_left1` for its classification.

The alternative `gov_*2` and `gov_*3` variables measure parliamentary
strength/support of governing parties and are outside the scope of this
analysis.

## Check where this belongs????


In [ ]:
# distribution/outlier analysis on the filtered dataset rather than the full historical dataset
df_period = df[df["year"].between(2000, 2019)].copy()

In [ ]:
df_period["inflation"].hist(bins=50)
plt.xlabel("Inflation (%)")
plt.ylabel("Frequency")
plt.title("Inflation Distribution, 2000–2019")

In [ ]:
df_period.boxplot(column="inflation")
plt.title("Inflation, 2000–2019")

In [ ]:
df_period.nlargest(10, "inflation")[
    ["country", "year", "inflation"]
]

In [ ]:
df_period[
    (df_period["country"] == "Romania") &
    (df_period["year"].between(1999, 2002))
][["country", "year", "inflation"]]

The inflation variable is strongly right-skewed. Most observations between 2000 and 2019 are concentrated at relatively low inflation rates, while several country-year observations show substantially higher inflation. The boxplot identifies these observations as statistical outliers according to the IQR rule, but they are not necessarily data errors and should not be removed without further investigation.

--
After verifying the data with the world data bank, we can conclude that 45.8% inflation in romania is an accurate value(even though its unusual, so we can keep it), since previos peack of inflation in 1993 was even bigger